# Training a Custom Piper TTS Voice

This notebook guides you through the step-by-step process of training a custom Text-to-Speech (TTS) voice for Piper. Piper is a fast, local neural text-to-speech system that utilizes the VITS architecture.

### Overview of the Workflow:
1. **Environment Setup**: Install system packages and compile alignment libraries.
2. **Dataset Preparation**: Organize audio files and transcripts in the LJSpeech format.
3. **Preprocessing**: Convert dataset files into training features.
4. **Fine-Tuning**: Fine-tune a pre-trained base model for fast convergence.
5. **Exporting to ONNX**: Convert PyTorch Lightning checkpoints to optimized ONNX models.
6. **Deployment**: Load the exported ONNX model directly into the `kb-tts` API.

## 1. Setup Environment

To train a Piper voice, you need system dependencies like `espeak-ng` (required for phonemization), standard compiler utilities, and PyTorch. If running locally, install `espeak-ng` and standard tools first.

In [ ]:
# System Dependencies (Linux/Debian/Ubuntu)
!sudo apt-get update && sudo apt-get install -y espeak-ng build-essential ffmpeg libasound2-dev libsndfile1-dev

### Clone and Build Piper Training Codebase

In [ ]:
# Clone rhasspy/piper repo containing the trainer module
!git clone https://github.com/rhasspy/piper.git ./piper

# Install training dependencies
%cd ./piper/src/python
!pip install --upgrade pip
!pip install cython
!pip install -r requirements.txt
!pip install -e .

# Compile the custom monotonic_align C++ cythonized extension
!chmod +x build_monotonic_align.sh
!./build_monotonic_align.sh
%cd ../../..

## 2. Prepare Dataset

The dataset should be in **LJSpeech format**:
- A folder `wav/` containing mono 22050Hz 16-bit PCM WAV audio files.
- A file `metadata.csv` mapping `filename_without_extension|text`.

You can generate this dataset automatically using the `kb-tts` Aggregator Web UI at `http://localhost:8000/training` or programmatically in Python as shown below:

In [ ]:
# Programmatic data aggregation using the kb-tts training module
import sys
import os

# Add project src directory to path
sys.path.append(os.path.abspath("../src"))
from kb_tts.training.data_generator import create_dataset_job, get_job
import time

youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ" # Replace with speaker audio URL
dataset_name = "rick_astley"

print("Starting dataset generation background task...")
job_id = create_dataset_job(youtube_url, dataset_name, language="en_US", whisper_model_size="base")

# Poll progress
while True:
    job = get_job(job_id)
    if not job:
        break
    print(f"Status: {job['status']} | Progress: {job['progress']}% | Segments: {job['num_segments']}")
    if job["status"] in ["completed", "failed"]:
        if job["status"] == "failed":
            print(f"Error: {job['error']}")
        break
    time.sleep(5)

## 3. Preprocessing

Before training, raw wave files and transcripts must be phonemized and formatted into processed features.

In [ ]:
# Setup output directories
!mkdir -p ./data/training_runs

# Run preprocessing script
!python3 -m piper_train.preprocess \
  --language en_US \
  --input-dir ./data/training_datasets/rick_astley \
  --output-dir ./data/training_runs/rick_astley_processed \
  --dataset-format ljspeech \
  --sample-rate 22050

## 4. Fine-Tuning

Rather than training from scratch (which takes weeks), we fine-tune a pre-trained base model. Download a checkpoint matching your target language and quality (e.g. English Medium model).

In [ ]:
# Download a pre-trained base checkpoint from Rhasspy Hugging Face
!wget https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/libritts/medium/epoch%3D3325-step%3D1164100.ckpt -O ./data/training_runs/libritts-medium.ckpt

In [ ]:
# Start training / fine-tuning
# Reduce batch-size if you encounter CUDA Out-Of-Memory (OOM) errors.
!python3 -m piper_train \
  --dataset-dir ./data/training_runs/rick_astley_processed \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --max_epochs 1000 \
  --resume_from_checkpoint ./data/training_runs/libritts-medium.ckpt

## 5. Exporting to ONNX

Once training reaches a point where the voice sounds good (typically after a few hundred epochs), export the PyTorch Lightning checkpoint to an ONNX file for fast runtime inference.

In [ ]:
# Find your latest checkpoint in the lightning_logs directory
# Example: ./data/training_runs/rick_astley_processed/lightning_logs/version_0/checkpoints/epoch=199-step=2000.ckpt

checkpoint_path = "./data/training_runs/rick_astley_processed/lightning_logs/version_0/checkpoints/epoch=199-step=2000.ckpt"
output_onnx = "./data/voices/rick_astley.onnx"

# Export to ONNX
!python3 -m piper_train.export_onnx \
  --checkpoint {checkpoint_path} \
  --output-file {output_onnx}

print(f"Successfully exported ONNX model to {output_onnx}")
print(f"A config file was also generated at {output_onnx}.json")

## 6. Deploy Custom Model

To deploy the custom model to your running `kb-tts` API instance:
1. Ensure the exported `.onnx` and `.onnx.json` files are placed inside the `data/voices/` directory.
2. Navigate to `http://localhost:8000/training` and choose the **Model Deployment** tab.
3. Map a new Voice Alias Key (e.g. `rick`) to the model `rick_astley`.
4. Test the model in the **TTS Playground** tab immediately!